# Stretch: more about errors and tests

For when the live exercise is done and there is time left, or for the week
after the course. These get harder as you go, so nobody is expected to
finish them all. Each one adds one idea.

Stretch 1 happens entirely in this notebook. For the others you try the
idea here first, then write a test for it in
`sessions/session-12/exercises/test_stretch.py`, which tests the functions
in `exercises/stretch.py`. Open both in VS Code next to this notebook.

In [ ]:
import importlib.util
import math
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

TESTS = "sessions/session-12/exercises/test_stretch.py"


def run_pytest(*args):
    """Run pytest with these arguments and print what it says, if it is installed."""
    if importlib.util.find_spec("pytest") is None:
        print("pytest is not installed in this Python:  python -m pip install pytest")
        return None
    result = subprocess.run([sys.executable, "-m", "pytest", *args, "-p", "no:cacheprovider"],
                            capture_output=True, text=True)
    print(result.stdout)
    return result.returncode

## Stretch 1: `else` and `finally`

A `try` statement can have two more parts. **Predict** what this prints for
`"4.5"` and for `"abc"` before you run the next cell. Write your
prediction in the markdown cell below first.

In [ ]:
def describe(text):
    """Print which parts of a try statement run for one piece of text."""
    try:
        number = float(text)
    except ValueError:
        print(f"  {text!r}: except  -> not a number")
    else:
        print(f"  {text!r}: else    -> got {number}")
    finally:
        print(f"  {text!r}: finally -> always")

*Your prediction here.*

In [ ]:
describe("4.5")
describe("abc")

Two questions to answer in the cell below:

1. Why put code in `else` rather than at the end of the `try` block? (Think
   about the live exercise's advice on how big a `try` should be.)
2. You have used something since session 4 that does the job of `finally`
   for files, without you writing it. What is it?

*Your answers here.*

## Stretch 2: raise your own error, with a better message

`average([])` says `ZeroDivisionError: division by zero`. That describes
what went wrong *inside* `average`. The person calling it did not divide
anything; they passed an empty list.

You met `raise` in the pytest walkthrough. Change `average` in
`exercises/stretch.py` so an empty list raises

```python
raise ValueError("average() of an empty list: there is nothing to average")
```

before it ever gets to the division. Try your version here first:

In [ ]:
def total(numbers):
    """Return the sum of a list of numbers."""
    running = 0
    for number in numbers:
        running += number
    return running


# TODO: def average(numbers):  that raises ValueError for an empty list


# average([])

Then the test, in `test_stretch.py`. `pytest.raises` can check the message
as well as the type:

```python
with pytest.raises(ValueError, match="empty list"):
    average([])
```

In [ ]:
run_pytest(TESTS, "-k", "average")

## Stretch 3: floats that are nearly equal

Your session 3 homework converted Celsius to Fahrenheit. Body temperature,
36.6°C, is 97.88°F. Or is it?

In [ ]:
def celsius_to_fahrenheit(c):
    """Return the Celsius temperature c converted to Fahrenheit."""
    return c * 9 / 5 + 32


print(celsius_to_fahrenheit(36.6))
print(celsius_to_fahrenheit(36.6) == 97.88)

Computers store most decimals as the nearest binary fraction, so tiny
errors creep in. The function is fine. The *test* is wrong to expect an
exact match.

Find another example of the same thing using `average` (hint: tenths), then
look up `math.isclose` and use it to compare the two numbers properly.

In [ ]:
# TODO: another float that is not quite what you expect


# TODO: math.isclose

In a test file the pytest version reads better:
`assert celsius_to_fahrenheit(36.6) == pytest.approx(97.88)`. Write a
failing test with plain `==` first, watch it fail, then fix it with
`pytest.approx`.

In [ ]:
run_pytest(TESTS, "-k", "fahrenheit or tenths")

## Stretch 4: one test, many cases

The live exercise had a separate test for every kind of messy minutes text.
When tests differ only in their data, pytest can run one test on a whole
table of cases:

```python
@pytest.mark.parametrize("text, expected", [
    ("4.65", 4.65),
    ("4.2 min", 4.2),
    # ...
])
def test_to_minutes_reads_numbers(text, expected):
    assert to_minutes(text) == expected
```

The `@` line is a **decorator**: it changes how the function below it is
used. Here it tells pytest to call the test once per row of the table.

Write it in `test_stretch.py` with at least five cases that should give a
number, and a second parametrized test for at least five that should give
`None`. Try to find a case that surprises you. (What does `to_minutes`
make of `"4,2"`, the way half of Europe writes four point two?)

In [ ]:
# Try cases out here first, before they go in the table

In [ ]:
run_pytest(TESTS, "-k", "minutes", "-v")

## Stretch 5: a test that needs a file

Testing `load_rows` on `data/clean/plays.csv` works, but the test then
depends on a 2,183-row file staying exactly the same. Better: a tiny file
made for the test, with rows whose right answer you know.

pytest will make you a fresh empty folder for each test. Ask for it by
naming an argument `tmp_path`:

```python
def test_load_rows_reads_a_small_file(tmp_path):
    path = tmp_path / "plays.csv"
    path.write_text("device,minutes_played\nphone,4.65\ncar,3.10\n", encoding="utf-8")
    rows = load_rows(path)
    assert len(rows) == 2
```

Things pytest hands to a test because of the argument's name are called
**fixtures**. Write that test, then add two more: a file with only a
header row, and a file that does not exist. What should each one do?

In [ ]:
run_pytest(TESTS, "-k", "load_rows", "-v")

## Stretch 6: pandas does not raise

Session 9 promised you would divide by a count that happens to be zero. In
plain Python that is a `ZeroDivisionError`. Watch what pandas does instead:

In [ ]:
plays = pd.read_csv("data/clean/plays.csv")

radio = plays[plays["device"] == "radio"]
print(len(radio), "plays on the radio")
print(radio["skipped"].mean())

No error at all. `nan`, "not a number", and it will flow quietly into
every calculation after it. There is nothing for `try` to catch, which
makes a test the only thing that will notice.

Write `skip_rate(plays, device)` in `exercises/stretch.py`: the percentage
of plays on that device that were skipped, or `None` if there are none.
Try it here first. The car should come out at 10.6%.

In [ ]:
# TODO: def skip_rate(plays, device):


# print(skip_rate(plays, "car"))
# print(skip_rate(plays, "radio"))

Then test it on a DataFrame you build by hand, four rows, where you can
work out every answer in your head:

```python
small = pd.DataFrame({
    "device": ["phone", "phone", "car", "car"],
    "skipped": [1, 0, 0, 0],
})
```

Phone: 1 of 2 skipped. Car: none. Radio: no plays at all. **This is how you
test a pandas function in your project**: never on the whole file, always
on three or four rows you made up on purpose.

If several tests need the same small DataFrame, look up `@pytest.fixture`:
it lets you write your own fixture, the way `tmp_path` is one.

In [ ]:
run_pytest(TESTS, "-k", "skip", "-v")

## All of them together

In [ ]:
run_pytest(TESTS)